# Just a test...

In [ ]:
import os
from dotenv import load_dotenv
import chromadb
from chromadb.utils import embedding_functions

load_dotenv()

chroma_client = chromadb.HttpClient(host='localhost', port="8000")

colls = chroma_client.list_collections()
colls

In [3]:
# Load
from langchain.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
data = loader.load()

from langchain.vectorstores import Chroma
# Split
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
all_splits = text_splitter.split_documents(data)

len(all_splits)

all_splits[4]

docs = [item.page_content for item in all_splits]
metas = [item.metadata for item in all_splits]

In [ ]:
data

In [4]:
bgelarge_ef = embedding_functions.HuggingFaceEmbeddingFunction(
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    model_name="BAAI/bge-base-en-v1.5"
)

In [5]:
chroma_vectorstore = Chroma(
    collection_name="my_collection", embedding_function=bgelarge_ef, client=chroma_client
)

retriever = chroma_vectorstore.as_retriever()



collection = chroma_client.get_or_create_collection(name="llama_collection", embedding_function=bgelarge_ef)
collection.count()
ids = list(range(len(all_splits)))

s_ids = [str(i) for i in ids]

In [ ]:
collection.add(ids=s_ids, documents=docs, metadatas=metas)
collection.count()

In [7]:
results = collection.query(query_texts="What is an agent", n_results=4)

In [ ]:
results